# 05 · Training

`NextSequence` owns the model, the loss, the optimizer and the training loop. You pass DataLoaders.

The first cell rebuilds the data from **02_data_preparation** in compact form.

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

import math
import torch
from torch.utils.data import DataLoader
from next_sequence import (CoreDataset, CustomCollate, Feature, NextSequence,
                           NextSequenceConfig, TaskType, create_bow)

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

def make_func(dtype):
    def func(values, inference=False):
        xs = [torch.tensor(v, dtype=dtype) for v in values]
        if inference:
            return xs, None
        return [x[:-1] for x in xs], [v[-1] for v in values]
    return func

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {"item": make_func(torch.long), "price": make_func(torch.float), "weekday": make_func(torch.long)}
MAX_LEN = 16

def loader(samples, inference=False, shuffle=False, batch_size=32):
    collate = CustomCollate(features, "item", funcs, inference=inference, max_len=MAX_LEN)
    return DataLoader(CoreDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

print(f"{len(users)} users | train/val/test samples: {len(train)}/{len(val)}/{len(test)} | vocab: {len(bow)}")

300 users | train/val/test samples: 300/300/300 | vocab: 7


## Training settings

| Argument | Default | Meaning |
|---|---|---|
| `epochs` | 10 | passes over the training loader |
| `lr` | 3e-4 | peak learning rate (AdamW, warmup then cosine decay) |
| `weight_decay` | 0.01 | on weight matrices only |
| `warmup_ratio` | 0.05 | share of steps used for warmup |
| `grad_clip` | 1.0 | max gradient norm (`None` to disable) |
| `device` | `None` | CUDA if available, then MPS, then CPU |
| `seed` | `None` | makes initialization and training reproducible |
| `warm_start` | `False` | a second `fit` continues instead of starting over |

In [2]:
config = NextSequenceConfig(features=features, max_len=MAX_LEN, emb_dim=32, attn_heads=2)
model = NextSequence(config, epochs=15, lr=3e-3, seed=0)
model

NextSequence(features=['item', 'price', 'weekday'], epochs=15, lr=0.003, device='cuda', not fitted)

## `fit`

With a validation loader, the weights of the epoch with the **lowest validation loss** are restored at the end.

In [3]:
model.fit(loader(train, shuffle=True), loader(val))
print("best epoch:", model.best_epoch_)

epoch=1 | train_loss=2.4787 | train_item=1.7906 | train_price=1.3915 | val_loss=1.9817 | val_item=1.5996 | val_price=0.8059
epoch=2 | train_loss=1.7599 | train_item=1.4127 | train_price=0.7457 | val_loss=1.7428 | val_item=1.3432 | val_price=0.8742
epoch=3 | train_loss=1.1985 | train_item=0.9809 | train_price=0.4814 | val_loss=0.9015 | val_item=0.7731 | val_price=0.2923


epoch=4 | train_loss=0.6655 | train_item=0.5860 | train_price=0.1799 | val_loss=0.5284 | val_item=0.4652 | val_price=0.1450
epoch=5 | train_loss=0.4345 | train_item=0.3808 | train_price=0.1243 | val_loss=0.2512 | val_item=0.2171 | val_price=0.0775
epoch=6 | train_loss=0.1962 | train_item=0.1592 | train_price=0.0859 | val_loss=0.2844 | val_item=0.2174 | val_price=0.1639


epoch=7 | train_loss=0.1236 | train_item=0.0999 | train_price=0.0805 | val_loss=0.0950 | val_item=0.0936 | val_price=0.0549
epoch=8 | train_loss=0.0401 | train_item=0.0503 | train_price=0.0401 | val_loss=0.0148 | val_item=0.0374 | val_price=0.0311
epoch=9 | train_loss=0.0067 | train_item=0.0313 | train_price=0.0362 | val_loss=-0.0086 | val_item=0.0303 | val_price=0.0229


epoch=10 | train_loss=-0.0191 | train_item=0.0217 | train_price=0.0261 | val_loss=-0.0216 | val_item=0.0258 | val_price=0.0240
epoch=11 | train_loss=-0.0341 | train_item=0.0179 | train_price=0.0209 | val_loss=-0.0307 | val_item=0.0254 | val_price=0.0206
epoch=12 | train_loss=-0.0394 | train_item=0.0163 | train_price=0.0241 | val_loss=-0.0352 | val_item=0.0248 | val_price=0.0209


epoch=13 | train_loss=-0.0455 | train_item=0.0152 | train_price=0.0212 | val_loss=-0.0423 | val_item=0.0218 | val_price=0.0177
epoch=14 | train_loss=-0.0488 | train_item=0.0147 | train_price=0.0191 | val_loss=-0.0440 | val_item=0.0213 | val_price=0.0171
epoch=15 | train_loss=-0.0482 | train_item=0.0142 | train_price=0.0217 | val_loss=-0.0441 | val_item=0.0213 | val_price=0.0171
best epoch: 15


## The history is a list of dicts, one per epoch

In [4]:
history = pd.DataFrame(model.history_).set_index("epoch")
history.round(4)

,train_loss,train_item,train_price,val_loss,val_item,val_price
epoch,,,,,,
1,2.4787,1.7906,1.3915,1.9817,1.5996,0.8059
2,1.7599,1.4127,0.7457,1.7428,1.3432,0.8742
3,1.1985,0.9809,0.4814,0.9015,0.7731,0.2923
4,0.6655,0.5860,0.1799,0.5284,0.4652,0.1450
5,0.4345,0.3808,0.1243,0.2512,0.2171,0.0775
6,0.1962,0.1592,0.0859,0.2844,0.2174,0.1639
7,0.1236,0.0999,0.0805,0.0950,0.0936,0.0549
8,0.0401,0.0503,0.0401,0.0148,0.0374,0.0311
9,0.0067,0.0313,0.0362,-0.0086,0.0303,0.0229


## Why the total loss goes negative

The tasks are balanced with learned uncertainty weighting
([Kendall et al., 2018](https://arxiv.org/abs/1705.07115)). Each task adds `0.5 · s`, where
`s = log σ²` is learned. As the model gets confident, `s` drops below zero and so does the total.
That is expected. **Compare the per-feature losses** (`train_item`, `val_price`, ...) instead.

The learned weights show how much each task counts:

In [5]:
model.task_weights()

{'item': 0.9729306697845459, 'price': 0.5971689224243164}

## Reproducibility: the same `seed` gives the same run

In [6]:
run_a = NextSequence(config, epochs=2, lr=3e-3, seed=42, verbose=False, device="cpu").fit(loader(train, shuffle=True))
run_b = NextSequence(config, epochs=2, lr=3e-3, seed=42, verbose=False, device="cpu").fit(loader(train, shuffle=True))
print("identical histories:", run_a.history_ == run_b.history_)

identical histories: True


## Continue training with `warm_start=True`

By default, calling `fit` again starts from a fresh model. With `warm_start=True` it continues
from the current weights, and the history keeps counting epochs.

In [7]:
more = NextSequence(config, epochs=3, lr=1e-3, seed=0, verbose=False, warm_start=True)
more.fit(loader(train, shuffle=True), loader(val))
more.fit(loader(train, shuffle=True), loader(val))
pd.DataFrame(more.history_)[["epoch", "train_loss", "val_loss"]].round(4)

,epoch,train_loss,val_loss
0,1,2.3994,2.0480
1,2,1.9420,1.9128
2,3,1.8203,1.8845
3,4,1.6844,1.6076
4,5,1.4012,1.4388
5,6,1.2789,1.3540
